In [ ]:
import logging
import os

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# local .py file
from scPRINT import (
    process_model,
    populate_lamin_db,
    SCPRINT_DEFS
)
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.constants import (
    FM_DEFS,
    FOUNDATION_MODEL_NAMES,
)
import numpy as np


In [ ]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scPRINT")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Get list of version keys from SCPRINT_DEFS
SCPRINT_VERSION_KEYS = list(SCPRINT_DEFS.VERSIONS.__dict__.keys())

In [ ]:
populate_lamin_db()

for version in SCPRINT_VERSION_KEYS:
    process_model(version, OUTPUT_DIR, MODEL_PATH)

In [ ]:
# Load results for a specific version (using MEDIUM as an example)
medium_version_id = SCPRINT_DEFS.VERSIONS.MEDIUM
file_prefix = f"{FOUNDATION_MODEL_NAMES.SCPRINT}_{medium_version_id}"
model = FoundationModel.load(OUTPUT_DIR, file_prefix)

GENES_OF_INTEREST = model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model.ordered_vocabulary]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_attn = model.weights.compute_attention_from_weights(
    layer_idx=5,
    n_heads=model.n_heads,
    vocab_mask=np.array(GENE_MASK)
)